# ANN с помощью библиотеки faiss

В этом тюториале мы научимся эффективно искать ближайших соседей с помощью библиотеки _faiss_.

Код и документация библиотеки: https://github.com/facebookresearch/faiss

Ее можно поставить в виртуальное окружение командой `pip install faiss-cpu`

Импортируем модули которые нам понадобятся впоследствии:

In [1]:
import logging
import pathlib
from timeit import default_timer as timer

import faiss
import gensim
import numpy as np

Подготовим логгер:

In [2]:
logging.basicConfig(level=logging.INFO)

## Данные

В качестве данных будем использовать предобученные эмбеддинги слов _word2vec_.

Домашняя страница: https://code.google.com/archive/p/word2vec/
Ссылка, по которой можно скачать предобученные векторы: https://drive.google.com/file/d/0B7XkCwpI5KDYNlNUTTlSS21pQmM/edit?usp=sharing

Посморим на уже скачанные данные:

In [3]:
!ls -lh models/word2vec

итого 3,4G
-rw-r--r-- 1 andrei andrei 3,4G апр 28 12:50 GoogleNews-vectors-negative300.bin


Загрузим вектора _word2vec_ с помощью библиотеки _gensim_:

In [4]:
w2v_model_file = "models/word2vec/GoogleNews-vectors-negative300.bin"

# Load word2vec embeddings
w2v_model = gensim.models.KeyedVectors.load_word2vec_format(w2v_model_file, binary=True)
keys = list(w2v_model.key_to_index.keys())
print(keys[:10])
logging.info(f"loaded embeddings: num_keys = {len(keys)} w2v_model_file = '{w2v_model_file}'")

INFO:gensim.models.keyedvectors:loading projection weights from models/word2vec/GoogleNews-vectors-negative300.bin
INFO:gensim.utils:KeyedVectors lifecycle event {'msg': 'loaded (3000000, 300) matrix of type float32 from models/word2vec/GoogleNews-vectors-negative300.bin', 'binary': True, 'encoding': 'utf8', 'datetime': '2026-04-30T19:57:32.247738', 'gensim': '4.4.0', 'python': '3.13.12 (main, Feb  4 2026, 15:06:39) [GCC 15.2.0]', 'platform': 'Linux-6.19.13+deb14-amd64-x86_64-with-glibc2.42', 'event': 'load_word2vec_format'}
INFO:root:loaded embeddings: num_keys = 3000000 w2v_model_file = 'models/word2vec/GoogleNews-vectors-negative300.bin'


['</s>', 'in', 'for', 'that', 'is', 'on', '##', 'The', 'with', 'said']


Получим из загруженной модели матрицу эмбеддингов:

In [5]:
E = w2v_model[keys]
logging.info(f"extracted embeddings: E.shape = {E.shape}")

INFO:root:extracted embeddings: E.shape = (3000000, 300)


## Индексация

Построим индекс _faiss_.

Существует множество разновидностей индексов, самые популярные:
- Flat-индексы, которые позволяют делать полный KNN
- IVF
- HNSW

У каждого индекса существует множество настроек, видов квантизации и т.п.
Подробнее про все эти индексы можно почитать по ссылке: https://github.com/facebookresearch/faiss/wiki/Faiss-indexes

In [6]:
# Index params
embedding_size = E.shape[1]
nlist = 100                   # number of cells for IVF index
metric = faiss.METRIC_L2      # alternative: faiss.METRIC_INNER_PRODUCT

start = timer()

# Create quantizer and index
quantizer = faiss.IndexFlatIP(embedding_size) # assigns vectors to Voronoi cells
faiss_index = faiss.IndexIVFFlat(quantizer, embedding_size, nlist, metric)

# Compute centroids
faiss_index.train(E)

# Populate index with embeddings
faiss_index.add(E)
logging.info(f"built faiss index: elapsed = {timer() - start:.3f}")

INFO:root:built faiss index: elapsed = 3.524


Пре необходимости, можем сохранить индекс в файл:

In [7]:
tmp_dir = pathlib.Path("tmp")
index_file = tmp_dir.joinpath(f"faiss.index")

# Save index
faiss.write_index(faiss_index, str(index_file))
logging.info(f"saved faiss index: index_file = '{index_file}'")

INFO:root:saved faiss index: index_file = 'tmp/faiss.index'


И загрузить его обратно:

In [8]:
# Load index
faiss_index = faiss.read_index(str(index_file))
logging.info(f"loaded faiss index: index_file = '{index_file}'")

INFO:root:loaded faiss index: index_file = 'tmp/faiss.index'


## Приближенный поиск ближайших соседей

Теперь можно попробовать поискать какое-нибудь слово в нашем индексе.

Тут мы столкнемся с двумя проблемами:
- нам понадобится эмбеддинг слова, которое мы будем искать. Будем считать, что наше слово есть в словаре, т.е. уже известно модели
- _faiss_ оперирует не словами, а ID слов. Эти ID назначаются словам в порядке добавления их в индекс, т.е. по сути являются номерами соответствующих словам строк в исходной матрице эмбеддмнгов 

Построим прямой и обратные словари слово <-> индекс слова:

In [9]:
# Keyword -> index vocabulary
key2idx = { key: idx for idx, key in enumerate(keys) }

# Keyword -> index reverse vocabulary
idx2key = { idx: key for key, idx in key2idx.items() }
print(list(idx2key.items())[:10])

[(0, '</s>'), (1, 'in'), (2, 'for'), (3, 'that'), (4, 'is'), (5, 'on'), (6, '##'), (7, 'The'), (8, 'with'), (9, 'said')]


Возьмем какое-нибудь ключевое слово и достанем из матрицы эмбеддингов его эмбеддинг:

In [10]:
# Get keyword index
key = "astronomy"
key_idx = key2idx[key]
logging.info(f"keyword found in vocabulary: key = '{key}' key_idx = {key_idx}")

# Get keyword embedding
e_key = E[key_idx]
print(e_key[:10])
logging.info(f"got keyword embedding: key = '{key}' e_key.shape = {e_key.shape}")

INFO:root:keyword found in vocabulary: key = 'astronomy' key_idx = 27223
INFO:root:got keyword embedding: key = 'astronomy' e_key.shape = (300,)


[ 0.20800781 -0.02124023  0.3671875   0.07421875  0.06835938  0.15625
  0.32617188 -0.328125    0.0189209   0.08496094]


Найдем ближайших соседей:

In [11]:
# Convert keyword embedding to list
E_keys = np.array([e_key])

# How many cells to look in
faiss_index.nprobe = 30

# Search
start = timer() 
D, I = faiss_index.search(E_keys, k=20)
print(I)
print(D)
logging.info(f"got k-NN: D.shape = {D.shape} I.shape = {I.shape} elapsed = {timer() - start:.3f}")

INFO:root:got k-NN: D.shape = (1, 20) I.shape = (1, 20) elapsed = 0.145


[[  27223   46036   80128 1343932 1539923  969709 1151358  720573 1417719
  1716710    1983  996889 1268645 1407483 1374760 1387747 1687945 1988829
   872842 1056582]]
[[0.        7.7235804 8.920389  8.931901  8.958224  9.083106  9.185755
  9.271894  9.357785  9.602903  9.604093  9.616962  9.64384   9.643951
  9.657419  9.689     9.711357  9.73758   9.747171  9.813942 ]]


На выходе получаем два массива:
- I -- это индексы найденных ближайших соседей
- D -- это расстояния этих соседей

Превратим индексы в слова с помощью подготовленных ранее словарей:

In [12]:
# Unpack found indexes and distances
idxs_knn = I[0]
dists_knn = D[0]
keys_knn = [idx2key[idx] for idx in idxs_knn]

# Print results
print(f"QUERY: key = {key}")
for i, key_knn in enumerate(keys_knn):
    dist = dists_knn[i]
    print(f"\tKNN: i = {i} key_knn = {key_knn} dist = {dist:.3f}")

QUERY: key = astronomy
	KNN: i = 0 key_knn = astronomy dist = 0.000
	KNN: i = 1 key_knn = Astronomy dist = 7.724
	KNN: i = 2 key_knn = astrophysics dist = 8.920
	KNN: i = 3 key_knn = celestial_observations dist = 8.932
	KNN: i = 4 key_knn = observational_astronomy dist = 8.958
	KNN: i = 5 key_knn = skywatching dist = 9.083
	KNN: i = 6 key_knn = Fraknoi dist = 9.186
	KNN: i = 7 key_knn = physics_astronomy dist = 9.272
	KNN: i = 8 key_knn = theoretical_astrophysics dist = 9.358
	KNN: i = 9 key_knn = constellations_planets dist = 9.603
	KNN: i = 10 key_knn = science dist = 9.604
	KNN: i = 11 key_knn = infrared_astronomy dist = 9.617
	KNN: i = 12 key_knn = heliophysics dist = 9.644
	KNN: i = 13 key_knn = astronomy_observatory dist = 9.644
	KNN: i = 14 key_knn = planetary_geology dist = 9.657
	KNN: i = 15 key_knn = astrophysical_phenomena dist = 9.689
	KNN: i = 16 key_knn = particle_astrophysics dist = 9.711
	KNN: i = 17 key_knn = Sidewalk_Astronomers dist = 9.738
	KNN: i = 18 key_knn = cel

Видим, что мы нашли слова, которые по смыслу близки к нашему ключевому слову.

Таким образом, мы:
- познакомились с библиотекой _faiss_
- научились использовать ее для приближенного поиска ближайших соседей

Дальше самостоятельно вы можете:
- попробовать разные виды индексов, и поиграться с их параметрами
- посчитать метрики качества, напр. какую долю из настоящего числа ближайших соседей находит ANN